# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library and is based on its Croissant metadata description.

### Dataset Source
The dataset is described via a Croissant schema available at the following public URL:

In [ ]:
# Install mlcroissant if needed
!pip install -q mlcroissant

## 1. Data Loading
We'll load the dataset and schema with `mlcroissant` and display metadata such as the dataset title and description. All references are via `@id` for reproducible processing.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata and create Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print("\nUse cases:")
for case in getattr(metadata, 'dataUseCases', []):
    print(f"- {case}")

## 2. Data Overview
Let's enumerate the available record sets and fields/columns using their `@id` keys, as described in the Croissant schema.

> **Note:** All entities (record sets, fields, columns) are referenced by their `@id` for consistent access and processing.

In [ ]:
# Enumerate all available record sets referenced by `@id`
record_sets = list(metadata.recordSet or getattr(metadata, 'recordSet', []))

if not record_sets:
    # Sometimes, record sets are nested inside distribution > hasPart in Croissant
    # We discover candidate record sets by introspection
    print("No direct 'recordSet' entries in metadata. Searching distributions for record set 'hasPart'...")
    record_sets = []
    for dist in getattr(metadata, 'distribution', []):
        if hasattr(dist, 'hasPart'):
            for rs in dist.hasPart:
                record_sets.append(rs)
if not record_sets:
    raise ValueError("No RecordSet definitions were found in the Croissant metadata.")

# RecordSet information by @id
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs._id}")
    field_ids = [f._id for f in getattr(rs, 'field', [])]
    print(f"  Fields: {', '.join(field_ids) if field_ids else '<none>'}")
    column_ids = [c._id for c in getattr(rs, 'column', [])]
    print(f"  Columns: {', '.join(column_ids) if column_ids else '<none>'}")

## 3. Data Extraction
We load data from the main record set (referenced by its `@id`), placing records into a Pandas DataFrame for analysis.

> **Tip:** Check the printed `@id`s above to select the primary record set and fields/columns for your analysis.

In [ ]:
# Define primary record set(s) by their @id
# Change this as needed based on your dataset's overview above
record_set_ids = [rs._id for rs in record_sets]

print(f"Loading data for RecordSet(s): {record_set_ids}")
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Successfully loaded: {record_set_id} [Columns: {list(df.columns)}]")
        print(df.head())
    except Exception as e:
        print(f"Could not load data for RecordSet {record_set_id}: {repr(e)}")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate several EDA steps: filtering, normalizing, and grouping on quantitative fields from our selected DataFrame. Use columns provided in the previous extraction or overview step.

Here, the protein dataset often includes fields such as `Coverage(%)`, `MW`, `Peptides`, or `Abundance`, which are commonly analyzed.

*Replace field IDs as necessary for your analysis.*

In [ ]:
# Pick a RecordSet/DataFrame to analyze
main_record_set_id = record_set_ids[0]  # Use the first found record set
df = dataframes[main_record_set_id]
print(f"Columns in {main_record_set_id}: {df.columns.tolist()}")

# EXAMPLE: Let's suppose the column '@id' for Coverage(%) is 'Coverage' and for MW 'MW'
# Find a likely numeric column
possible_numeric = [col for col in df.columns if df[col].dtype in ('float64', 'int64') or col.lower() in ('coverage', 'mw', 'peptides', 'abundance', 'normalized_abundance', 'molecular_weight')]

if not possible_numeric:
    # Try to convert columns to numeric if possible
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
        except Exception:
            continue
    possible_numeric = [col for col in df.columns if df[col].dtype in ('float64', 'int64')]

if possible_numeric:
    numeric_field = possible_numeric[0]
else:
    raise RuntimeError('No numeric field found for EDA.')
print(f"Using numeric field for filtering: {numeric_field}")

threshold = df[numeric_field].mean()  # Use mean as dynamic threshold example
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)}")

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a common string/categorical field, such as 'Sample' or 'Category' (check columns for a suitable group field)
possible_group_fields = [c for c in df.columns if df[c].dtype=='O' and c.lower() not in ['description', 'name']]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped mean normalized {numeric_field} by {group_field}:")
    print(grouped_df[[f"{numeric_field}_normalized"]].head())

## 5. Visualization
Let's visualize a distribution of the main numeric field and its normalized values, as well as show grouped means (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field], kde=True, color='steelblue')
plt.title(f'Distribution of {numeric_field} (>mean)')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, color='crimson')
plt.title(f'Normalized Distribution of {numeric_field} (>mean)')
plt.xlabel(f"{numeric_field}_normalized")
plt.ylabel('Count')
plt.show()

# Group mean plot if grouping field exists
if group_field:
    grouped_vals = grouped_df[f"{numeric_field}_normalized"]
    plt.figure(figsize=(10,5))
    grouped_vals.plot(kind='bar')
    plt.title(f'Group mean of normalized {numeric_field} by {group_field}')
    plt.ylabel(f'{numeric_field}_normalized (mean)')
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
- We successfully loaded and explored the FAIR^2 mass spectrometry dataset via its Croissant schema using only `@id` references for record sets and fields.
- Numerical field distributions were explored and visualized, and we demonstrated field normalization and groupwise aggregation for deeper insights.
- For additional processing, you may join with related metadata tables, create composite features, or export cleaned data for downstream machine learning analysis.